In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
from google.colab import userdata
api_key = userdata.get('Gemini_FileSearch_API')

In [6]:
from google import genai
from google.genai import types
import time

client = genai.Client(api_key=api_key)

# Creating the FileStore
file_search_store = client.file_search_stores.create(
    config={
        'display_name': 'TryingFileStore'
    }
)


In [16]:
#getting the files path from GoogleDrive
import os
doc_path = "/content/drive/MyDrive/Colab Notebooks/Docs"
files = [os.path.join(doc_path, f) for f in os.listdir(doc_path) if os.path.isfile(os.path.join(doc_path, f))]
files

['/content/drive/MyDrive/Colab Notebooks/Docs/7.01 Game Plan for Multimodal RAG.txt',
 '/content/drive/MyDrive/Colab Notebooks/Docs/7.02 Introduction to Multimodal RAG.txt',
 '/content/drive/MyDrive/Colab Notebooks/Docs/7.03 Python - Sep up and Video.txt']

In [34]:
#uploading files to file store
for file in files:
  operation = client.file_search_stores.upload_to_file_search_store(
    file=file,
    file_search_store_name=file_search_store.name,
    config={
        'display_name' : os.path.basename(file), #prints what files are uploaded
    }
  )

  while not operation.done:
      time.sleep(5)
      operation = client.operations.get(operation)


KeyboardInterrupt: 

In [45]:
# deleting the accidently twice uploaded files
dele = ['fileSearchStores/my-file_search-store-123/documents/my_doc', 'fileSearchStores/my-file_search-store-123/documents/my_doc1'] #original been removesd, this are examples
for delete in dele:
  client.file_search_stores.delete(name=delete, config={'force': True})

In [47]:
response = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents="""Tell me about multimodal rag""",
    config=types.GenerateContentConfig(
        tools=[
            types.Tool(
                file_search=types.FileSearch(
                    file_search_store_names=[file_search_store.name]
                )
            )
        ]
    )
)

print(response.text)

**Multimodal Retrieval-Augmented Generation (RAG)** is an advanced evolution of traditional RAG that allows AI systems to retrieve and process information from multiple formats—such as **text, images, audio, and video**—instead of being limited to text alone.

While traditional RAG typically converts text documents into vectors for retrieval, Multimodal RAG creates a bridge between different types of data so that a text-based query can find a relevant image, or a video clip can be used to generate a detailed text summary.

---

### 1. How It Works: The Core Architecture
The primary challenge of Multimodal RAG is getting different types of data (like a JPG image and a PDF paragraph) to "speak the same language." This is achieved through three main stages:

*   **Shared Embedding Spaces (The Bridge):**
    *   To make multimodal data searchable, systems use models like **CLIP (Contrastive Language-Image Pre-training)** or **ImageBind**. 
    *   These models are trained to map different 